# Day 2 — Dataset inspection and preparation
**Before using any dataset, read its card and licence.** Do not assume text datasets have paired audio.
- COMI-LINGUA (`LID` config, CC-BY-4.0): token-level Hindi/English/other labels — text only.
- CoSHE-500 / IITG-HingCoS: candidate Hinglish speech for ASR evaluation (verify access & terms; request HingCoS early).
- Own recordings: consented speakers, manually verified transcripts.

In [ ]:
# --- Setup (run once per Colab session) ---
REPO_URL = "https://github.com/arkalatua/Indic_Switch.git"
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.exists("/content/Indic_Switch"):
        subprocess.run(["git", "clone", REPO_URL, "/content/Indic_Switch"], check=True)
    %cd /content/Indic_Switch
    !pip install -q -r requirements.txt
    # Optional: persist models/results across sessions
    # from google.colab import drive; drive.mount("/content/drive")
else:
    os.chdir(os.path.abspath(".."))  # running locally from notebooks/
sys.path.insert(0, os.getcwd())

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## Inspect COMI-LINGUA

In [ ]:
from datasets import load_dataset
ds = load_dataset("LingoIITGN/COMI-LINGUA", "LID")
print(ds)
ex = ds["train"][0]
for k, v in ex.items():
    print(f"--- {k}\n{str(v)[:400]}")

In [ ]:
from src.data_sources import comi_row_to_example
from collections import Counter
parsed = [comi_row_to_example(r) for r in ds["train"].select(range(2000))]
ok = [p for p in parsed if p]
print(f"parsed {len(ok)}/{len(parsed)}; annotators agreeing:", Counter(p["n_annotators"] for p in ok))
print("tag distribution:", Counter(t for p in ok for t in p["tags"]))
print(list(zip(ok[0]["tokens"], ok[0]["tags"])))

## Write leakage-safe LID splits

In [ ]:
!python scripts/prepare_comi_lingua.py
import json; print(json.dumps(json.load(open("data/processed/lid_stats.json")), indent=2)[:1500])

## Build an ASR evaluation manifest
Inspect the dataset's columns first, then set `--text-col` / `--speaker-col` accordingly.
Splits are speaker-independent and near-duplicate-safe (`src.preprocessing.assign_splits`).

In [ ]:
from datasets import load_dataset_builder
b = load_dataset_builder("Trelis/CoSHE-500")
print(b.info.description[:800] if b.info.description else ""); print(b.info.features)

In [ ]:
# CoSHE-500: 500 clips (6.9-29.5 s), columns audio/transcription, Hindi in Devanagari + English in Latin.
# No speaker ids -> pure evaluation set (--all-test). Licence CC-BY-NC-4.0: non-commercial, attribute.
!python scripts/build_manifest.py hf --dataset Trelis/CoSHE-500 --split eval --text-col transcription --name coshe500 --licence "CC-BY-NC-4.0 (soketlabs/CoSHE-Eval)" --all-test

In [ ]:
# Own consented recordings: data/raw/own/*.wav + transcripts.csv (file,transcript,speaker_id)
# !python scripts/build_manifest.py folder --audio-dir data/raw/own --transcripts data/raw/own/transcripts.csv --name own_recordings --licence "consented; not redistributable"

In [ ]:
import pandas as pd
m = pd.read_csv("data/manifests/coshe500.csv"); display(m.head()); print(m.split.value_counts(), m.duration.describe())